In [ ]:
import os
import sys
import time
import subprocess
import ctypes.util

IS_LINUX = sys.platform.startswith("linux")
if IS_LINUX:
    os.environ["MUJOCO_GL"] = "osmesa"
    os.environ["PYOPENGL_PLATFORM"] = "osmesa"
else:
    os.environ["MUJOCO_GL"] = "glfw"

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--quiet", "--only-binary=:all:",
        "gymnasium[mujoco]>=1.2,<2",
        "mujoco>=3.3,<4",
        "imageio", "imageio-ffmpeg", "pillow", "pandas", "matplotlib"
    ],
    check=True
)

if IS_LINUX and not ctypes.util.find_library("OSMesa"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "libosmesa6", "libgl1"], check=True)

import numpy as np
import gymnasium as gym
import mujoco
from IPython.display import display, Video

from walker_env import make_env

print(f"Python {sys.executable}")
print(f"Gymnasium {gym.__version__} | MuJoCo {mujoco.__version__}")

In [ ]:
for folder in ("results", "figures", "videos"):
    os.makedirs(folder, exist_ok=True)

import json
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

torch.set_num_threads(1)

ALGORITHMS = ["ddpg", "td3", "sac"]
TRAIN_TYPES = ["nominal", "dr"]
TRAIN_SEEDS = [0, 1, 2]
TRAIN_RANGE = (1.0, 2.0)
MASS_SCALES = [0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5]
HELD_OUT_MASSES = [m for m in MASS_SCALES if not TRAIN_RANGE[0] <= m <= TRAIN_RANGE[1]]
STATE_SEEDS = list(range(5000, 5100))
REFERENCE_SEEDS = list(range(100, 105))
OOD_STATE_SEEDS = STATE_SEEDS[:20]
OOD_MASSES = [1.0, 1.5, 2.0, 2.5]

HEIGHT_MIN = 0.8
ANGLE_LIMIT = 1.0
SATURATION_LEVEL = 0.95
PRE_FALL_STEPS = 50

COLORS = {"ddpg": "tab:blue", "td3": "tab:orange", "sac": "tab:green"}
LABELS = {"ddpg": "DDPG", "td3": "TD3", "sac": "SAC"}
TRAIN_LABELS = {"nominal": "trained at x1.0", "dr": "domain randomised"}
LINESTYLES = {"nominal": "-", "dr": "--"}

probe_env = make_env()
state_dim = probe_env.observation_space.shape[0]
action_dim = probe_env.action_space.shape[0]
dt = float(probe_env.unwrapped.dt)
JOINT_NAMES = [probe_env.unwrapped.model.joint(i).name for i in range(3, 3 + action_dim)]
probe_env.close()

OBS_NAMES = (
    ["torso height", "torso angle"]
    + [f"{j} angle" for j in JOINT_NAMES]
    + ["forward velocity", "vertical velocity", "angular velocity"]
    + [f"{j} velocity" for j in JOINT_NAMES]
)

print(f"state_dim={state_dim} | action_dim={action_dim} | dt={dt}s")
print(f"Masses {MASS_SCALES} | held out from DR training range {TRAIN_RANGE}: {HELD_OUT_MASSES}")

In [ ]:
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim),
            nn.Tanh(),
        )

    def forward(self, state):
        return self.network(state)


LOG_STD_MIN, LOG_STD_MAX = -20, 2


class GaussianActor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
        )
        self.mean_head = nn.Linear(256, action_dim)
        self.log_std_head = nn.Linear(256, action_dim)

    def forward(self, state):
        return torch.tanh(self.mean_head(self.body(state)))

In [ ]:
def policy_tag(algo, training, seed):
    return f"{algo}_dr_seed{seed}" if training == "dr" else f"{algo}_seed{seed}"


def load_policy(algo, path):
    actor = GaussianActor(state_dim, action_dim) if algo == "sac" else Actor(state_dim, action_dim)
    actor.load_state_dict(torch.load(path, weights_only=True))
    actor.eval()
    return actor


policies = {}
for algo in ALGORITHMS:
    for training in TRAIN_TYPES:
        for seed in TRAIN_SEEDS:
            path = f"models/{policy_tag(algo, training, seed)}.pt"
            if os.path.exists(path):
                policies[(algo, training, seed)] = load_policy(algo, path)

if not policies:
    raise RuntimeError("No trained policies found in models/. Run ./run_all.sh first.")

found = pd.DataFrame([{"algorithm": a, "training": t, "seed": s} for a, t, s in policies])
print(f"Loaded {len(policies)} policies")
display(found.groupby(["algorithm", "training"])["seed"].apply(list).unstack())

In [ ]:
def failure_reason(height, angle, terminated):
    if not terminated:
        return "survived"
    if angle > ANGLE_LIMIT:
        return "tipped forward"
    if angle < -ANGLE_LIMIT:
        return "tipped backward"
    if height < HEIGHT_MIN:
        return "collapsed"
    return "too high"


def run_episode(actor, torso_mass_scale, state_seed, record=False):
    env = make_env(torso_mass_scale=torso_mass_scale)
    try:
        obs, _ = env.reset(seed=state_seed)
        data = env.unwrapped.data
        start_x = float(data.qpos[0])
        observations, heights, angles, actions = [obs.copy()], [], [], []
        total_reward, steps = 0.0, 0
        terminated = truncated = False

        while not (terminated or truncated):
            with torch.no_grad():
                action = actor(torch.as_tensor(obs, dtype=torch.float32)).numpy()
            obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += float(reward)
            steps += 1
            observations.append(obs.copy())
            heights.append(float(data.qpos[1]))
            angles.append(float(data.qpos[2]))
            actions.append(action)

        heights, angles, actions = np.array(heights), np.array(angles), np.array(actions)
        saturated = np.abs(actions) > SATURATION_LEVEL
        survival_s = steps * dt
        distance = float(data.qpos[0]) - start_x
        result = {
            "torso_mass_scale": torso_mass_scale,
            "state_seed": state_seed,
            "torso_mass_kg": env.current_torso_mass,
            "total_mass_kg": env.current_total_mass,
            "survived": not terminated,
            "failure": failure_reason(heights[-1], angles[-1], terminated),
            "steps": steps,
            "survival_s": survival_s,
            "distance_m": distance,
            "avg_speed_mps": distance / survival_s,
            "return": total_reward,
            "mean_height_m": float(heights.mean()),
            "mean_pitch_rad": float(angles.mean()),
            "pitch_std_rad": float(angles.std()),
            "mean_effort": float(np.mean(actions ** 2)),
            "height_margin_m": float(heights.min() - HEIGHT_MIN),
            "angle_margin_rad": float(ANGLE_LIMIT - np.abs(angles).max()),
            "forward_margin_rad": float(ANGLE_LIMIT - angles.max()),
            "backward_margin_rad": float(ANGLE_LIMIT + angles.min()),
            "saturation": float(saturated.mean()),
            "saturation_last_0.4s": float(saturated[-PRE_FALL_STEPS:].mean()),
        }
        if record:
            result["trajectory"] = {
                "observations": np.array(observations),
                "heights": heights,
                "angles": angles,
                "actions": actions,
            }
        return result
    finally:
        env.close()

In [ ]:
print("1. Mass is applied")
for scale in MASS_SCALES:
    env = make_env(torso_mass_scale=scale)
    print(f"   x{scale:<5} torso {env.current_torso_mass:6.3f} kg | total {env.current_total_mass:6.3f} kg")
    env.close()

key = next(iter(policies))
first = run_episode(policies[key], 1.5, STATE_SEEDS[0])
second = run_episode(policies[key], 1.5, STATE_SEEDS[0])
same = first["distance_m"] == second["distance_m"]
print(f"\n2. Determinism ({key}): {first['distance_m']:+.4f} m vs {second['distance_m']:+.4f} m -> {'PASS' if same else 'FAIL'}")

print("\n3. Saved weights reproduce the distances recorded at the end of training (x1.0)")
mismatches = 0
for (algo, training, seed), actor in policies.items():
    tag = policy_tag(algo, training, seed)
    with open(f"models/{tag}_reference.json") as f:
        reference = {int(k): v for k, v in json.load(f).items()}
    errors = [abs(run_episode(actor, 1.0, s)["distance_m"] - reference[s]) for s in REFERENCE_SEEDS]
    ok = max(errors) < 1e-6
    mismatches += not ok
    print(f"   {tag:16} max error {max(errors):.2e} m -> {'PASS' if ok else 'FAIL'}")

if not same or mismatches:
    print("\nSOME CHECKS FAILED: if only check 3 fails, train.py ran under a different Python than this kernel.")
else:
    print("\nAll checks passed.")

In [ ]:
start = time.time()
rows = []
for (algo, training, seed), actor in policies.items():
    for scale in MASS_SCALES:
        for state_seed in STATE_SEEDS:
            result = run_episode(actor, scale, state_seed)
            result.update(algorithm=algo, training=training, train_seed=seed)
            rows.append(result)
    print(f"{policy_tag(algo, training, seed):16} done | {time.time() - start:.0f}s elapsed")

results = pd.DataFrame(rows)
POLICY_KEYS = ["algorithm", "training", "train_seed"]
walks_at_1x = (
    results[results["torso_mass_scale"] == 1.0][POLICY_KEYS + ["state_seed", "survived"]]
    .rename(columns={"survived": "walks_at_1x"})
)
results = results.merge(walks_at_1x, on=POLICY_KEYS + ["state_seed"])
results.to_csv("results/multiseed_results.csv", index=False)
print(f"\n{len(results)} episodes saved to results/multiseed_results.csv")

In [ ]:
METRICS = [
    "survival", "survival_given_1x", "distance_m", "avg_speed_mps", "mean_height_m",
    "mean_pitch_rad", "height_margin_m", "angle_margin_rad", "forward_margin_rad",
    "backward_margin_rad", "saturation", "mean_effort",
]
GROUP = ["algorithm", "training", "torso_mass_scale"]

per_policy = (
    results.groupby(POLICY_KEYS + ["torso_mass_scale"])
    .agg(
        survival=("survived", "mean"),
        distance_m=("distance_m", "mean"),
        avg_speed_mps=("avg_speed_mps", "mean"),
        mean_height_m=("mean_height_m", "mean"),
        mean_pitch_rad=("mean_pitch_rad", "mean"),
        height_margin_m=("height_margin_m", "mean"),
        angle_margin_rad=("angle_margin_rad", "mean"),
        forward_margin_rad=("forward_margin_rad", "mean"),
        backward_margin_rad=("backward_margin_rad", "mean"),
        saturation=("saturation", "mean"),
        mean_effort=("mean_effort", "mean"),
    )
    .reset_index()
)
conditional = (
    results[results["walks_at_1x"]]
    .groupby(POLICY_KEYS + ["torso_mass_scale"])["survived"].mean()
    .rename("survival_given_1x").reset_index()
)
per_policy = per_policy.merge(conditional, on=POLICY_KEYS + ["torso_mass_scale"], how="left")
per_policy.to_csv("results/multiseed_per_policy.csv", index=False)

aggregations = {"n_seeds": ("train_seed", "nunique")}
for metric in METRICS:
    for stat in ("mean", "std", "min", "max"):
        aggregations[f"{metric}_{stat}"] = (metric, stat)
summary = per_policy.groupby(GROUP).agg(**aggregations).reset_index()
summary.to_csv("results/multiseed_summary.csv", index=False)


def mean_spread_table(metric, scale=1.0, digits=0):
    table = summary.assign(
        cell=lambda d: (d[f"{metric}_mean"] * scale).round(digits).astype(str)
        + " ± " + (d[f"{metric}_std"].fillna(0) * scale).round(digits).astype(str)
    )
    return table.pivot(index="torso_mass_scale", columns=["algorithm", "training"], values="cell")


print("Survival (%) from 100 shared starting states, mean ± std across training seeds")
display(mean_spread_table("survival", 100))
print("Survival (%) among starts the same policy walks at x1.0")
display(mean_spread_table("survival_given_1x", 100))
print("Distance walked (m)")
display(mean_spread_table("distance_m", 1, 1))

In [ ]:
PANELS = [
    ("survival", "Survival, all starts", 100, "%"),
    ("survival_given_1x", "Survival, starts it walks at x1.0", 100, "%"),
    ("distance_m", "Distance walked", 1, "m"),
    ("height_margin_m", "Height margin to 0.8 m limit", 1, "m"),
    ("angle_margin_rad", "Pitch margin to the ±1 rad limit", 1, "rad"),
    ("mean_pitch_rad", "Mean pitch (+ = leaning forward)", 1, "rad"),
    ("saturation", "Torques at their limit", 100, "% of actions"),
    ("avg_speed_mps", "Average speed", 1, "m/s"),
    ("mean_effort", "Control effort", 1, "mean action²"),
]

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for ax, (metric, title, scale, unit) in zip(axes.flat, PANELS):
    ax.axvspan(*TRAIN_RANGE, color="gray", alpha=0.08)
    for algo in ALGORITHMS:
        for training in TRAIN_TYPES:
            d = summary[(summary["algorithm"] == algo) & (summary["training"] == training)].sort_values("torso_mass_scale")
            if d.empty:
                continue
            ax.plot(d["torso_mass_scale"], d[f"{metric}_mean"] * scale, LINESTYLES[training], marker="o", markersize=4,
                    color=COLORS[algo], label=f"{LABELS[algo]}, {TRAIN_LABELS[training]}")
            ax.fill_between(d["torso_mass_scale"], d[f"{metric}_min"] * scale, d[f"{metric}_max"] * scale,
                            color=COLORS[algo], alpha=0.12)
    if metric in ("height_margin_m", "angle_margin_rad"):
        ax.axhline(0, color="red", linewidth=1, linestyle=":")
    ax.axvline(1.0, color="gray", linestyle=":", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Torso mass scale")
    ax.set_ylabel(unit)
    ax.set_xticks(MASS_SCALES)
    ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=9)
fig.suptitle(
    f"Payload sweep: line = mean over training seeds, band = min to max across seeds; "
    f"grey = DR training range x{TRAIN_RANGE[0]:g} to x{TRAIN_RANGE[1]:g}",
    fontsize=14,
)
plt.tight_layout()
fig.savefig("figures/multiseed_payload.png", dpi=150)
plt.show()

In [ ]:
REASONS = ["tipped forward", "tipped backward", "collapsed", "too high"]
REASON_COLORS = {"tipped forward": "tab:red", "tipped backward": "tab:purple", "collapsed": "tab:brown", "too high": "tab:gray"}

shares = (
    results.groupby(GROUP)["failure"].value_counts(normalize=True)
    .unstack(fill_value=0).reindex(columns=["survived"] + REASONS, fill_value=0) * 100
)
shares.round(0).to_csv("results/failure_reasons.csv")

trained = [t for t in TRAIN_TYPES if t in results["training"].unique()]
fig, axes = plt.subplots(len(trained), len(ALGORITHMS), figsize=(18, 4.5 * len(trained)), squeeze=False)
for row, training in enumerate(trained):
    for col, algo in enumerate(ALGORITHMS):
        ax = axes[row, col]
        if (algo, training) not in shares.index.droplevel(2):
            ax.set_visible(False)
            continue
        d = shares.loc[(algo, training)]
        bottom = np.zeros(len(d))
        x = np.arange(len(d))
        for reason in REASONS:
            ax.bar(x, d[reason], bottom=bottom, color=REASON_COLORS[reason], label=reason)
            bottom += d[reason].to_numpy()
        ax.set_xticks(x, [f"x{m:g}" for m in d.index])
        ax.set_ylim(0, 100)
        ax.set_ylabel("% of episodes that fell")
        ax.set_title(f"{LABELS[algo]}, {TRAIN_LABELS[training]}")
        ax.grid(alpha=0.3, axis="y")
axes[0, 0].legend()
fig.suptitle("How each policy falls (share of all episodes, by cause)", fontsize=14)
plt.tight_layout()
fig.savefig("figures/failure_reasons.png", dpi=150)
plt.show()

fall_profile = (
    results.groupby(GROUP + ["survived"])
    .agg(
        episodes=("survived", "size"),
        median_time_s=("survival_s", "median"),
        saturation=("saturation", "mean"),
        saturation_last_04s=("saturation_last_0.4s", "mean"),
        forward_margin=("forward_margin_rad", "mean"),
        backward_margin=("backward_margin_rad", "mean"),
        height_margin=("height_margin_m", "mean"),
    )
    .round(3)
)
fall_profile.to_csv("results/fall_profile.csv")
print("Fallers vs survivors at x1.0, x2.0 and x2.5 (saturation_last_04s = share of torques at the limit just before the end)")
display(fall_profile[fall_profile.index.get_level_values("torso_mass_scale").isin([1.0, 2.0, 2.5])])

In [ ]:
def explain_failure(algo, training, seed, mass, state_seed=None):
    actor = policies[(algo, training, seed)]
    if state_seed is None:
        candidates = results[
            (results["algorithm"] == algo) & (results["training"] == training) & (results["train_seed"] == seed)
            & (results["torso_mass_scale"] == mass) & results["walks_at_1x"] & ~results["survived"]
        ]
        if candidates.empty:
            print(f"{policy_tag(algo, training, seed)} never falls at x{mass:g} from a start it walks at x1.0.")
            return
        state_seed = int(candidates["state_seed"].iloc[0])

    loaded = run_episode(actor, mass, state_seed, record=True)
    normal = run_episode(actor, 1.0, state_seed, record=True)
    tl, tn = loaded["trajectory"], normal["trajectory"]
    time_l = np.arange(1, len(tl["heights"]) + 1) * dt
    time_n = np.arange(1, len(tn["heights"]) + 1) * dt

    fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
    axes[0].plot(time_n, tn["heights"], color="gray", linestyle="--", label="x1.0, same start")
    axes[0].plot(time_l, tl["heights"], color="black", label=f"x{mass:g}")
    axes[0].axhline(HEIGHT_MIN, color="red", linestyle=":")
    axes[0].set_ylabel("Torso height (m)")
    axes[0].legend(loc="lower left")
    axes[1].plot(time_n, tn["angles"], color="gray", linestyle="--")
    axes[1].plot(time_l, tl["angles"], color="black")
    for limit in (ANGLE_LIMIT, -ANGLE_LIMIT):
        axes[1].axhline(limit, color="red", linestyle=":")
    axes[1].set_ylabel("Pitch (rad, + = forward)")
    for j, name in enumerate(JOINT_NAMES):
        axes[2].plot(time_l, tl["actions"][:, j], linewidth=1, label=name)
    axes[2].axhspan(SATURATION_LEVEL, 1.05, color="red", alpha=0.08)
    axes[2].axhspan(-1.05, -SATURATION_LEVEL, color="red", alpha=0.08)
    axes[2].set_ylabel(f"Joint torque at x{mass:g}")
    axes[2].set_xlabel("Time (s)")
    axes[2].legend(ncol=3, fontsize=9, loc="lower left")
    for ax in axes:
        ax.grid(alpha=0.3)
        if not loaded["survived"]:
            ax.axvline(time_l[-1], color="red", alpha=0.5)
    tag = policy_tag(algo, training, seed)
    title = f"{tag} at x{mass:g}, start {state_seed}: {loaded['failure']} after {loaded['survival_s']:.2f} s"
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    fig.savefig(f"figures/failure_{tag}_x{mass:g}_start{state_seed}.png", dpi=150)
    plt.show()

    sat_joint = (np.abs(tl["actions"][-PRE_FALL_STEPS:]) > SATURATION_LEVEL).mean(axis=0)
    worst = JOINT_NAMES[int(np.argmax(sat_joint))]
    print(title)
    print(f"  Pitch range: x1.0 {tn['angles'].min():+.2f} to {tn['angles'].max():+.2f} rad | "
          f"x{mass:g} {tl['angles'].min():+.2f} to {tl['angles'].max():+.2f} rad")
    print(f"  Lowest torso: x1.0 {tn['heights'].min():.2f} m | x{mass:g} {tl['heights'].min():.2f} m")
    print(f"  Torques at the limit: x1.0 {normal['saturation']:.0%} overall | x{mass:g} {loaded['saturation']:.0%} overall, "
          f"{loaded['saturation_last_0.4s']:.0%} in the final 0.4 s (most saturated joint: {worst}, {sat_joint.max():.0%})")


for algo in ALGORITHMS:
    if (algo, "nominal", TRAIN_SEEDS[0]) in policies:
        explain_failure(algo, "nominal", TRAIN_SEEDS[0], 2.0)

In [ ]:
def collect_states(actor, scale, seeds):
    return np.concatenate([run_episode(actor, scale, s, record=True)["trajectory"]["observations"] for s in seeds])


start = time.time()
ood_rows, ood_dims = [], []
for (algo, training, seed), actor in policies.items():
    reference = collect_states(actor, 1.0, OOD_STATE_SEEDS)
    low, high = np.percentile(reference, [1, 99], axis=0)
    for scale in OOD_MASSES:
        states = reference if scale == 1.0 else collect_states(actor, scale, OOD_STATE_SEEDS)
        outside = (states < low) | (states > high)
        ood_rows.append({
            "algorithm": algo, "training": training, "train_seed": seed, "torso_mass_scale": scale,
            "ood_state_fraction": float(outside.any(axis=1).mean()),
            "ood_dim_fraction": float(outside.mean()),
        })
        ood_dims.append(pd.Series(outside.mean(axis=0), index=OBS_NAMES, name=(algo, training, seed, scale)))
    print(f"{policy_tag(algo, training, seed):16} done | {time.time() - start:.0f}s elapsed")

ood = pd.DataFrame(ood_rows)
ood.to_csv("results/ood_states.csv", index=False)
ood_by_dim = pd.DataFrame(ood_dims)
ood_by_dim.index.names = ["algorithm", "training", "train_seed", "torso_mass_scale"]
ood_by_dim.to_csv("results/ood_by_dimension.csv")
ood_summary = ood.groupby(GROUP).agg(
    state_mean=("ood_state_fraction", "mean"), state_min=("ood_state_fraction", "min"), state_max=("ood_state_fraction", "max"),
    dim_mean=("ood_dim_fraction", "mean"),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, column, title in [
    (axes[0], "state", "States with any variable outside the x1.0 range"),
    (axes[1], "dim", "Share of state variables outside the x1.0 range"),
]:
    for algo in ALGORITHMS:
        for training in TRAIN_TYPES:
            d = ood_summary[(ood_summary["algorithm"] == algo) & (ood_summary["training"] == training)]
            if d.empty:
                continue
            ax.plot(d["torso_mass_scale"], d[f"{column}_mean"] * 100, LINESTYLES[training], marker="o",
                    color=COLORS[algo], label=f"{LABELS[algo]}, {TRAIN_LABELS[training]}")
            if column == "state":
                ax.fill_between(d["torso_mass_scale"], d["state_min"] * 100, d["state_max"] * 100, color=COLORS[algo], alpha=0.12)
    ax.set_title(title)
    ax.set_xlabel("Torso mass scale")
    ax.set_ylabel("%")
    ax.set_xticks(OOD_MASSES)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=9)
fig.suptitle("Out-of-distribution states (x1.0 range = each policy's own 1st to 99th percentile at normal mass)", fontsize=13)
plt.tight_layout()
fig.savefig("figures/ood_states.png", dpi=150)
plt.show()

print("At x2.0, the state variables pushed furthest outside the normal-mass range (mean over seeds):")
heavy = ood_by_dim[ood_by_dim.index.get_level_values("torso_mass_scale") == 2.0]
for (algo, training), group in heavy.groupby(level=["algorithm", "training"]):
    top = group.mean().sort_values(ascending=False).head(3)
    print(f"  {LABELS[algo]:4} {TRAIN_LABELS[training]:18} " + ", ".join(f"{name} {share:.0%}" for name, share in top.items()))

In [ ]:
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFont

STRIDE = 8
TILE_W, TILE_H = 400, 240
GRID_COLS = 3
VIDEO_MASSES = [0.75, 1.0, 1.5, 2.0, 2.5]
try:
    FONT = ImageFont.load_default(size=16)
except TypeError:
    FONT = ImageFont.load_default()
video_fps = 1 / (STRIDE * dt)


def record_tiles(actor, scale, state_seed):
    env = make_env(torso_mass_scale=scale, render_mode="rgb_array")
    tiles = []
    try:
        obs, _ = env.reset(seed=state_seed)
        data = env.unwrapped.data
        start_x = float(data.qpos[0])
        env.render()
        camera = env.unwrapped.mujoco_renderer.viewer.cam
        camera.azimuth, camera.elevation, camera.distance = 90.0, -10.0, 4.0

        def capture(steps):
            camera.lookat[:] = [data.body("torso").xpos[0], 0.0, 1.0]
            image = Image.fromarray(env.render()).resize((TILE_W, TILE_H))
            tiles.append((image, steps, float(data.qpos[0]) - start_x))

        capture(0)
        steps = 0
        terminated = truncated = False
        while not (terminated or truncated):
            with torch.no_grad():
                action = actor(torch.as_tensor(obs, dtype=torch.float32)).numpy()
            obs, _, terminated, truncated, _ = env.step(action)
            steps += 1
            if steps % STRIDE == 0 or terminated or truncated:
                capture(steps)
        return tiles, bool(terminated)
    finally:
        env.close()


def label_tile(image, label, scale, steps, distance, fell, finished):
    tile = image.copy()
    draw = ImageDraw.Draw(tile)
    draw.rectangle((0, 0, TILE_W, 26), fill=(22, 26, 34))
    draw.text((8, 4), f"{label} x{scale} | {distance:+.1f} m | {steps * dt:.1f}s", fill="white", font=FONT)
    if finished:
        status, color = ("FELL", (255, 90, 90)) if fell else ("FINISHED", (120, 230, 120))
        draw.text((TILE_W - 90, 4), status, fill=color, font=FONT)
    return tile


def make_gait_video(algo, training, seed, state_seed, mass_scales=VIDEO_MASSES):
    actor = policies[(algo, training, seed)]
    tag = policy_tag(algo, training, seed)
    label = f"{LABELS[algo]}{' DR' if training == 'dr' else ''}"
    grid_rows = -(-len(mass_scales) // GRID_COLS)
    rollouts = {scale: record_tiles(actor, scale, state_seed) for scale in mass_scales}
    n_frames = max(len(tiles) for tiles, _ in rollouts.values())
    path = f"videos/gait_{tag}_start{state_seed}.mp4"

    with imageio.get_writer(path, fps=video_fps, codec="libx264", macro_block_size=16) as writer:
        for i in range(n_frames):
            canvas = Image.new("RGB", (GRID_COLS * TILE_W, grid_rows * TILE_H))
            for k, scale in enumerate(mass_scales):
                tiles, fell = rollouts[scale]
                last = len(tiles) - 1
                image, steps, distance = tiles[min(i, last)]
                tile = label_tile(image, label, scale, steps, distance, fell, finished=i >= last)
                canvas.paste(tile, ((k % GRID_COLS) * TILE_W, (k // GRID_COLS) * TILE_H))
            writer.append_data(np.asarray(canvas))

    outcomes = ", ".join(f"x{s}: {'fell' if f else 'finished'} at {t[-1][2]:+.1f} m" for s, (t, f) in rollouts.items())
    print(f"{tag} (start {state_seed}) -> {path}\n  {outcomes}")
    display(Video(path, embed=True, width=900))


video_policies = [k for k in policies if k[2] == TRAIN_SEEDS[0]]
walks = results[(results["torso_mass_scale"] == 1.0) & (results["train_seed"] == TRAIN_SEEDS[0])]
common = walks.groupby("state_seed")["survived"].all()
VIDEO_STATE = int(common[common].index[0]) if common.any() else STATE_SEEDS[0]
print(f"Video start state: {VIDEO_STATE}")
for algo, training, seed in video_policies:
    make_gait_video(algo, training, seed, VIDEO_STATE)